# Video Watcher — run it from your phone

Watch and understand a TikTok / YouTube / Instagram video from your phone's browser, free.

**Setup (easiest on a phone):** paste your `sk-ant-...` key into the **API_KEY** box in Cell 2, then run. (Or add it under the key icon / Secrets in the left sidebar.) For a big speed-up, set **Runtime > Change runtime type > T4 GPU** (free).

Then paste a video link into Cell 3 and tap **Runtime > Run all**.

> Your key is your password to a paid account. Never paste it into a chat or message. It only goes in the box below. Don't use **File > Save a copy** with your key filled in, or it gets saved into the copy.

In [ ]:
# Cell 1 — install dependencies (ffmpeg is required; install it if missing)
!pip -q install -U "faster-whisper" anthropic "yt-dlp" curl_cffi
!which ffmpeg >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y ffmpeg)

In [ ]:
# Cell 2 — fetch the script, load your API key, detect a GPU
# Easiest on a phone: paste your sk-ant-... key into the API_KEY box, then run.
API_KEY = ""  #@param {type:"string"}
import os, subprocess, urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/Chudi1475/video-watcher/main/video_watcher.py",
    "video_watcher.py")
key = API_KEY.strip()
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY") or ""
    except Exception:
        key = ""
if not key:
    raise SystemExit(
        "Paste your sk-ant-... key into the API_KEY box above, then re-run "
        "this cell (or add it under the key icon / Secrets).")
os.environ["ANTHROPIC_API_KEY"] = key
try:
    has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    has_gpu = False
DEVICE = ["--device", "cuda", "--compute-type", "float16"] if has_gpu else []
print("GPU available:", has_gpu)

In [ ]:
# Cell 3 — paste a video link, then download it for free with yt-dlp
URL = ""  #@param {type:"string"}
import subprocess

def _dl(extra):
    cmd = ["yt-dlp", "--no-playlist", "-f", "bv*+ba/b",
           "--merge-output-format", "mp4", "-o", "input.%(ext)s",
           "--print", "after_move:filepath"] + extra + [URL]
    return subprocess.run(cmd, capture_output=True, text=True)

if not URL.strip():
    raise SystemExit("Paste a video URL into the URL field above, then re-run.")
r = _dl(["--impersonate", "chrome"])   # helps TikTok / Instagram
if r.returncode != 0:
    r = _dl([])                          # retry without impersonation
if r.returncode != 0:
    raise SystemExit("download failed:\n" + (r.stderr or "")[-1500:])
src = r.stdout.strip().splitlines()[-1]
print("downloaded:", src)

In [ ]:
# Cell 4 — run the tool (--fast --social; uses the GPU if one was found)
import subprocess
subprocess.run(["python", "video_watcher.py", src, "--fast", "--social"] + DEVICE,
               check=True)

In [ ]:
# Cell 5 — show the result inline
import os, glob
from IPython.display import Markdown, display
stem = os.path.splitext(os.path.basename(src))[0]
path = os.path.join(stem + "_watch", "understanding.md")
if not os.path.exists(path):
    cands = sorted(glob.glob("*_watch/understanding.md"))
    path = cands[0] if cands else None
if path and os.path.exists(path):
    display(Markdown(open(path, encoding="utf-8").read()))
else:
    print("understanding.md not found; check the output of Cell 4 above.")

### Notes

- **Instagram** public links sometimes return a rate-limit / login error on Colab. If so, upload the video file straight into the Colab file pane (folder icon, left), then set `src = "yourfile.mp4"` and run Cells 4 and 5 only.
- yt-dlp extractors change often. If a site stops working mid-session, run a cell with `!yt-dlp -U`.
- On the free CPU runtime, keep clips short or rely on the `--fast` preset. Switching to a free **T4 GPU** (Runtime > Change runtime type) makes transcription much faster.
- The output files live in the `*_watch` folder (open the file pane to download them).